## Cleaning steps

1. Remove duplicates
2. Remove all row nulls
3. Fill the nulls/''/' '  
4. Trim spaces
5. Standardize case
6. Cast data types
7. Filter invalid rows
7. Rename columns
9. Add metadata columns if applicable
10. Validate schema

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType


## Load table

In [0]:
df = spark.table('bikes_catalog.bronze.bronze_loc_a101')
df.show()

## Cleaning

In [0]:
df = df.dropDuplicates()

In [0]:
df = df.dropna(how='all')

In [0]:
for col in df.columns:
    print(f'Nulls in {col}: {df.filter(F.col(col).isNull()).count()}') 

In [0]:
df = df.na.fill({'CNTRY': 'Unknown'})
df.show()

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name,F.trim(F.col(field.name)))

df.show()

In [0]:
df.select('CNTRY').distinct().show()

In [0]:
df = df.withColumn(
    'CNTRY',
    F.when(df.CNTRY == '', 'Unknown')
    .when(((df.CNTRY == 'USA') | (df.CNTRY == 'United States')), 'US')
    .when(df.CNTRY == 'United Kingdom', 'UK')
    .when(df.CNTRY == 'DE', 'Germany')
    .otherwise(df.CNTRY)
)

df.show()

In [0]:
df.select('CNTRY').distinct().show()

In [0]:
df = df.withColumn('CID', F.regexp_replace('CID', '-', ''))
df.show()

In [0]:
df = df.withColumnRenamed('CNTRY','Country')
df.show()

In [0]:
df.printSchema()

In [0]:
df.write.mode('overwrite').saveAsTable('bikes_catalog.silver.silver_loc_a101')